In [3]:
import os
import glob
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import numpy as np
import random
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import pprint
import pyspark
import pyspark.sql.functions as F

from pyspark.sql.functions import col
from pyspark.sql.types import StringType, IntegerType, FloatType, DateType

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer, f1_score, roc_auc_score
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

import model_inference


In [2]:
# Build a .py script that takes a snapshot date, loads a model artefact and make an inference and save to datamart

## set up pyspark session

In [4]:
# Initialize SparkSession
spark = pyspark.sql.SparkSession.builder \
    .appName("dev") \
    .master("local[*]") \
    .getOrCreate()

# Set log level to ERROR to hide warnings
spark.sparkContext.setLogLevel("ERROR")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/31 11:15:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## set up config

In [5]:
snapshot_date_str = "2024-01-01"
model_name = "credit_model_2024_09_01.pkl"


In [6]:
config = {}
config["snapshot_date_str"] = snapshot_date_str
config["snapshot_date"] = datetime.strptime(config["snapshot_date_str"], "%Y-%m-%d")
config["model_name"] = model_name
config["model_bank_directory"] = "model_bank/"
config["model_artefact_filepath"] = config["model_bank_directory"] + config["model_name"]

pprint.pprint(config)

{'model_artefact_filepath': 'model_bank/credit_model_2024_09_01.pkl',
 'model_bank_directory': 'model_bank/',
 'model_name': 'credit_model_2024_09_01.pkl',
 'snapshot_date': datetime.datetime(2024, 1, 1, 0, 0),
 'snapshot_date_str': '2024-01-01'}


## load model artefact from model bank

In [8]:
# Load the model from the pickle file
with open(config["model_artefact_filepath"], 'rb') as file:
    model_artefact = pickle.load(file)

print("Model loaded successfully! " + config["model_artefact_filepath"])

Model loaded successfully! model_bank/credit_model_2024_09_01.pkl


## load feature store

In [9]:
# --- load feature store ---
folder_path = "datamart/gold/feature_store/"
files_list = [folder_path+os.path.basename(f) for f in glob.glob(os.path.join(folder_path, '*'))]
features_store_sdf = spark.read.option("header", "true").parquet(*files_list)
features_store_sdf = features_store_sdf.withColumnRenamed(
    "snapshot_date","feature_snapshot_date"
)

print("row_count:",features_store_sdf.count())


# extract feature store
features_sdf = features_store_sdf.filter((col("feature_snapshot_date") == config["snapshot_date"]))
print("extracted features_sdf", features_sdf.count(), config["snapshot_date"])

features_pdf = features_sdf.toPandas()
features_pdf

row_count: 9984


extracted features_sdf 485 2024-01-01 00:00:00


,loan_id,customer_id,feature_snapshot_date,age,occupation,annual_income,monthly_inhand_salary,num_bank_accounts,num_credit_card,interest_rate,...,outstanding_debt,credit_utilization_ratio,payment_of_min_amount,total_emi_per_month,amount_invested_monthly,monthly_balance,credit_history_age_year,credit_history_age_month,payment_behaviour_spent,payment_behaviour_value
0,CUS_0x102d_2024_01_01,CUS_0x102d,2024-01-01,31.0,Entrepreneur,89064.523438,7256.043457,5.0,3.0,1.0,...,648.359985,30.574299,No,37.572750,296.094116,641.937439,30,3,High,Medium
1,CUS_0x1051_2024_01_01,CUS_0x1051,2024-01-01,42.0,Engineer,35022.218750,2859.518311,3.0,5.0,4.0,...,1000.440002,25.390232,No,21.214577,259.345123,295.392151,28,6,Low,Small
2,CUS_0x1269_2024_01_01,CUS_0x1269,2024-01-01,22.0,Manager,42031.089844,3762.590820,2.0,1.0,8.0,...,83.550003,24.065590,No,66.506180,108.855858,450.897034,17,3,High,Medium
3,CUS_0x1290_2024_01_01,CUS_0x1290,2024-01-01,31.0,Architect,10455.875000,729.322937,6.0,5.0,12.0,...,83.160004,29.464912,No,25.748447,24.490505,302.693329,23,3,Low,Medium
4,CUS_0x12d1_2024_01_01,CUS_0x12d1,2024-01-01,41.0,Accountant,21384.939453,1646.078369,4.0,4.0,15.0,...,691.530029,24.884464,Yes,22.095625,168.245499,254.266708,27,0,Low,Medium
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
480,CUS_0xc68d_2024_01_01,CUS_0xc68d,2024-01-01,41.0,Musician,15497.019531,1193.418335,8.0,4.0,19.0,...,1180.560059,35.188225,Yes,24.166683,107.718864,267.456299,15,11,Low,Medium
481,CUS_0xc6d8_2024_01_01,CUS_0xc6d8,2024-01-01,34.0,Developer,120955.921875,10021.660156,2.0,7.0,4.0,...,369.359985,37.368149,No,57.505169,241.974396,942.686401,21,4,High,Large
482,CUS_0xe8d_2024_01_01,CUS_0xe8d,2024-01-01,26.0,Journalist,14441.509766,1453.459106,9.0,8.0,26.0,...,1336.310059,40.085510,Yes,21.031950,48.389664,325.924316,13,6,High,Medium
483,CUS_0xf9e_2024_01_01,CUS_0xf9e,2024-01-01,43.0,Teacher,142475.312500,12013.943359,8.0,5.0,11.0,...,628.570007,42.344322,No,124.400009,81.778755,1235.215576,30,8,High,Large


## preprocess data for modeling

In [10]:
# prepare X_inference
exclude_cols = [
"customer_id", "feature_snapshot_date"
]
feature_cols = [c for c in features_pdf.columns if c not in exclude_cols]
X_inference = features_pdf[feature_cols].copy()

# Identify categorical columns
cat_cols = X_inference.select_dtypes(include=['object']).columns.tolist()

# Handle categorical encoding
for col in cat_cols:
    # Try to reuse training mappings if stored; otherwise, auto-map
    try:
        mapping = model_artefact["preprocessing_transformers"].get(f"{col}_mapping", None)
        if mapping is not None:
            X_inference[col] = X_inference[col].map(mapping)
        else:
            # fallback: derive from inference data
            X_inference[col] = X_inference[col].astype('category').cat.codes
    except Exception as e:
        print(f"Warning: Could not map column {col} ({e}), fallback to category codes.")
        X_inference[col] = X_inference[col].astype('category').cat.codes

# Replace NaN / inf values
X_inference.replace([np.inf, -np.inf], np.nan, inplace=True)
X_inference.fillna(X_inference.mean(), inplace=True)

# apply transformer - standard scaler
X_inference = X_inference.astype(float)
transformer_stdscaler = model_artefact["preprocessing_transformers"]["stdscaler"]
X_inference = transformer_stdscaler.transform(X_inference)

print('X_inference', X_inference.shape[0])
X_inference

ValueError: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- credit_history_age_year
- credit_mix
- loan_id
- occupation
- payment_behaviour_spent
- ...
Feature names seen at fit time, yet now missing:
- auto_loan
- avg_fe_1
- avg_fe_10
- avg_fe_11
- avg_fe_12
- ...


## model prediction inference

In [ ]:
 features_pdf

In [10]:
# load model
model = model_artefact["model"]

# predict model
y_inference = model.predict_proba(X_inference)[:, 1]

# prepare output
y_inference_pdf = features_pdf[["customer_id","feature_snapshot_date"]].copy()
y_inference_pdf["model_name"] = config["model_name"]
y_inference_pdf["model_predictions"] = y_inference
y_inference_pdf

,customer_id,feature_snapshot_date,model_name,model_predictions
0,CUS_0x133e,2024-01-01,credit_model_2024_09_01.pkl,0.032150
1,CUS_0x14d0,2024-01-01,credit_model_2024_09_01.pkl,0.227160
2,CUS_0x1668,2024-01-01,credit_model_2024_09_01.pkl,0.029446
3,CUS_0x18cb,2024-01-01,credit_model_2024_09_01.pkl,0.107939
4,CUS_0x1eee,2024-01-01,credit_model_2024_09_01.pkl,0.053485
...,...,...,...,...
480,CUS_0xbf9c,2024-01-01,credit_model_2024_09_01.pkl,0.116170
481,CUS_0xc131,2024-01-01,credit_model_2024_09_01.pkl,0.930701
482,CUS_0xc208,2024-01-01,credit_model_2024_09_01.pkl,0.043068
483,CUS_0xc50f,2024-01-01,credit_model_2024_09_01.pkl,0.232881


## save model inference to datamart gold table

In [11]:
# create bronze datalake
gold_directory = f"datamart/gold/model_predictions/{config["model_name"][:-4]}/"
print(gold_directory)

if not os.path.exists(gold_directory):
    os.makedirs(gold_directory)

# save gold table - IRL connect to database to write
partition_name = config["model_name"][:-4] + "_predictions_" + snapshot_date_str.replace('-','_') + '.parquet'
filepath = gold_directory + partition_name
spark.createDataFrame(y_inference_pdf).write.mode("overwrite").parquet(filepath)
# df.toPandas().to_parquet(filepath,
#           compression='gzip')
print('saved to:', filepath)

datamart/gold/model_predictions/credit_model_2024_09_01/


saved to: datamart/gold/model_predictions/credit_model_2024_09_01/credit_model_2024_09_01_predictions_2024_01_01.parquet


## backfill

In [11]:
# set up config
snapshot_date_str = "2023-01-01"

start_date_str = "2023-01-01"
end_date_str = "2024-12-01"

In [12]:
# generate list of dates to process
def generate_first_of_month_dates(start_date_str, end_date_str):
    # Convert the date strings to datetime objects
    start_date = datetime.strptime(start_date_str, "%Y-%m-%d")
    end_date = datetime.strptime(end_date_str, "%Y-%m-%d")
    
    # List to store the first of month dates
    first_of_month_dates = []

    # Start from the first of the month of the start_date
    current_date = datetime(start_date.year, start_date.month, 1)

    while current_date <= end_date:
        # Append the date in yyyy-mm-dd format
        first_of_month_dates.append(current_date.strftime("%Y-%m-%d"))
        
        # Move to the first of the next month
        if current_date.month == 12:
            current_date = datetime(current_date.year + 1, 1, 1)
        else:
            current_date = datetime(current_date.year, current_date.month + 1, 1)

    return first_of_month_dates

dates_str_lst = generate_first_of_month_dates(start_date_str, end_date_str)


In [13]:
for snapshot_date in dates_str_lst:
    print(snapshot_date)
    model_inference.main(snapshot_date, model_name)

2023-01-01


---starting job---


{'model_artefact_filepath': 'model_bank/credit_model_2024_09_01.pkl',
 'model_bank_directory': 'model_bank/',
 'model_name': 'credit_model_2024_09_01.pkl',
 'snapshot_date': datetime.datetime(2023, 1, 1, 0, 0),
 'snapshot_date_str': '2023-01-01'}
Model loaded successfully! model_bank/credit_model_2024_09_01.pkl
row_count: 9984
extracted features_sdf 530 2023-01-01 00:00:00


ValueError: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- credit_history_age_year
- credit_mix
- loan_id
- occupation
- payment_behaviour_spent
- ...
Feature names seen at fit time, yet now missing:
- auto_loan
- avg_fe_1
- avg_fe_10
- avg_fe_11
- avg_fe_12
- ...


## Check datamart

In [14]:
# Initialize SparkSession
spark = pyspark.sql.SparkSession.builder \
    .appName("dev") \
    .master("local[*]") \
    .getOrCreate()

# Set log level to ERROR to hide warnings
spark.sparkContext.setLogLevel("ERROR")

In [15]:
folder_path = "datamart/gold/model_predictions/credit_model_2024_09_01/"
files_list = [folder_path+os.path.basename(f) for f in glob.glob(os.path.join(folder_path, '*'))]
df = spark.read.option("header", "true").parquet(*files_list)
print("row_count:",df.count())

df.show()

AnalysisException: [UNABLE_TO_INFER_SCHEMA] Unable to infer schema for Parquet. It must be specified manually.